In [ ]:
get_ipython().system('pip install settrade-v2')
get_ipython().system('pip install schedule')
get_ipython().system('pip install requests')

import os
import schedule
import time
from settrade_v2 import Investor
import datetime
import re
import logging
import requests
import concurrent.futures
from IPython.display import clear_output # Import clear_output

# --- การกำหนดค่าการบันทึกข้อมูล (Logging Configuration) ---
# ตั้งค่าให้ Logging แสดงเวลาเป็นประเทศไทย (UTC+7) เสมอ
def thai_time_converter(*args):
    utc_now = datetime.datetime.now(datetime.timezone.utc)
    thai_time = utc_now + datetime.timedelta(hours=7)
    return thai_time.timetuple()

logging.Formatter.converter = thai_time_converter

# 1. ตั้งค่า Logger หลักของระบบ
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    force=True, # บังคับเขียนทับการตั้งค่าเดิม (สำคัญสำหรับการรันใน Jupyter)
    handlers=[
        logging.FileHandler("trade_bot.log"),
        logging.StreamHandler()
    ]
)

# 2. บังคับตั้งค่า Logger ของ settradev2 โดยเฉพาะ
settrade_logger = logging.getLogger("settradev2")
settrade_logger.setLevel(logging.INFO)
settrade_logger.propagate = True # บังคับให้ส่งข้อความออกหน้าจอร่วมกับระบบหลัก
# --- สิ้นสุดการกำหนดค่าการบันทึกข้อมูล ---

# --- การกำหนดค่าควบคุมการแสดงผลเพื่อความเร็ว ---
SHOW_TRADING_DECISION = False # ตั้งค่าเป็น True False หากต้องการดูรายละเอียดการคำนวณและตัดสินใจแบบ Real-time
# ----------------------------------------

# --- การกำหนดค่า Telegram Bot ---
TELEGRAM_BOT_TOKEN = "8884529549:AAHoJOKtX0BNNlJ220EJCBDOba-hUrong60"
TELEGRAM_CHAT_ID = "8738487946"

def send_telegram_message(message):
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        logging.warning("ยังไม่ได้ตั้งค่า TELEGRAM_BOT_TOKEN หรือ TELEGRAM_CHAT_ID. จะไม่ส่งข้อความ Telegram.")
        return

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "HTML"
    }
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        logging.info(f"ส่งข้อความ Telegram สำเร็จ: {message}")
    except requests.exceptions.RequestException as e:
        logging.error(f"เกิดข้อผิดพลาดในการส่งข้อความ Telegram: {e}")
# --- สิ้นสุดการกำหนดค่า Telegram Bot ---

# --- Helper function to get Thai time object ---
def get_thai_time_object():
    utc_now = datetime.datetime.now(datetime.timezone.utc)
    thai_time = utc_now + datetime.timedelta(hours=7)
    return thai_time # This should return a datetime object, not timetuple()

def get_thai_time():
    return get_thai_time_object().strftime('%Y-%m-%d %H:%M:%S')
# --- End Helper function to get Thai time object ---

# --- Daily Resource File Download ---
# This section adds a new scheduled task to download a file daily after 16:35.
# The filename will include the current date and time to ensure uniqueness.
def download_resources_file():
    """Downloads a file from a specified URL and saves it with a timestamp."""
    # TODO: Replace with the actual URL of the file you want to download
    file_url = "https://www.example.com/your_resource.csv"

    # Generate a timestamp for the filename
    thai_now = get_thai_time_object()
    timestamp_str = thai_now.strftime("%Y-%m-%d_%H-%M-%S")

    # TODO: Customize the base filename as needed
    filename = f"resources_{timestamp_str}.csv"

    try:
        response = requests.get(file_url, stream=True)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        logging.info(f"ไฟล์ '{filename}' ดาวน์โหลดสำเร็จ.")
        send_telegram_message(f"✅ <b>ดาวน์โหลดไฟล์</b>: '{filename}' สำเร็จ!")
    except requests.exceptions.RequestException as e:
        logging.error(f"เกิดข้อผิดพลาดในการดาวน์โหลดไฟล์ '{file_url}': {e}")
        send_telegram_message(f"❌ <b>ดาวน์โหลดไฟล์</b>: เกิดข้อผิดพลาดในการดาวน์โหลดไฟล์ '{file_url}' ({e})")

# --- End Daily Resource File Download ---

# --- Global flag to control bot execution ---
stop_bot_flag = False

def stop_bot_gracefully():
    global stop_bot_flag
    stop_bot_flag = True
    logging.info("ตั้งค่าธงหยุดการทำงานของบอต. บอตจะหยุดในรอบการทำงานถัดไป.")
    send_telegram_message("🛑 <b>บอตหยุดทำงาน</b>: บอตจะหยุดการทำงานตามกำหนดเวลา 12:31 น.")

# 1. ตั้งค่า Investor
investor = Investor(
    app_id="xkWnqI4xysxO5Pwm",
    app_secret="AMlBm8BXUPFg2IfOZE3B5918fJa9Kvi1Q15asw5lkx9m",
    broker_id="023",
    app_code="ALGO_EQ",
    is_auto_queue = False
)

market = investor.MarketData()
equity = investor.Equity(account_no="6724266")
account_no = "6724266"

# --- กำหนดค่าระบบควบคุมเวลา ---
startup_delay_seconds_after_market_open = 1
run_immediately_regardless_of_market_hours = True
R_MULTIPLE_WARNING_THRESHOLD = 0.8

# Helper function to get quote with retries
def get_quote_with_retries(symbol, retries=3, delay=1, timeout=5):
    for i in range(retries):
        try:
            quote = market.get_quote_symbol(symbol)
            if quote and 'last' in quote and quote['last'] is not None:
                return quote
        except requests.exceptions.ConnectTimeout:
            logging.warning(f"Connect Timeout for {symbol}. Retrying in {delay} seconds... (Attempt {i+1}/{retries})")
            time.sleep(delay)
        except Exception as e:
            logging.error(f"Error fetching quote for {symbol}: {e}")
            break # Exit on other errors
    logging.error(f"Failed to get quote for {symbol} after {retries} attempts.")
    return None

# 2. รายการหุ้นที่ต้องการติดตาม
stocks_to_monitor = [
  #  {
  #       'symbol': 'SET5013C2609C',
  #       'volume': None ,
  #       'amount_invested': 0,
  #       'status': 'bought',
  #       'actual_volume_bought': 30500,
  #       'await_buy_at_support': False,
  #       'trailing_stop_percentage': 0,
  #       'trailing_stop_price': 0.31,
  #       'fixed_stop_loss_percent': 0,
  #       'buy_at_price': '0',
  #       'buy_at_price1_5': '0',
  #       'buy_at_price2_5': '0',
  #       'buy2_at_price': '0',
  #       'buy_runtrend': '0',
  #       'sell_at_price': '0',
  #       'buy_price': 0.34,
  #       'custom_high': None,
  #       'custom_low': None,
  #       'buy_trigger_time': None
  #   },
       # 1
    {
        'symbol': 'WHA',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 4.94,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '5.05',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 2
    {
        'symbol': 'WHAUP',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 7.75,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '7.90',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 3
    {
        'symbol': 'TTB',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 3.10,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '3.16',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 4
    # {
    #     'symbol': 'TTA',
    #     'volume': None ,
    #     'amount_invested': 30000,
    #     'status': 'pending',
    #     'actual_volume_bought': 0,
    #     'await_buy_at_support': False,
    #     'trailing_stop_percentage': 0,
    #     'trailing_stop_price': 5.85,
    #     'fixed_stop_loss_percent': 0,
    #     'buy_at_price': '0',
    #     'buy_at_price1_5': '0',
    #     'buy_at_price2_5': '0',
    #     'buy2_at_price': '0',
    #     'buy_runtrend': '6.00',
    #     'sell_at_price': '0',
    #     'buy_price': 0,
    #     'custom_high': None,
    #     'custom_low': None,
    #     'buy_trigger_time': None
    # },
    # 5
    {
        'symbol': 'TRUE',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 13.40,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '13.70',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 6
    {
        'symbol': 'TQM',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 16.30,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '16.60',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 7
    # {
    #     'symbol': 'TOP',
    #     'volume': None ,
    #     'amount_invested': 30000,
    #     'status': 'pending',
    #     'actual_volume_bought': 0,
    #     'await_buy_at_support': False,
    #     'trailing_stop_percentage': 0,
    #     'trailing_stop_price': 64.25,
    #     'fixed_stop_loss_percent': 0,
    #     'buy_at_price': '0',
    #     'buy_at_price1_5': '0',
    #     'buy_at_price2_5': '0',
    #     'buy2_at_price': '0',
    #     'buy_runtrend': '65.00',
    #     'sell_at_price': '0',
    #     'buy_price': 0,
    #     'custom_high': None,
    #     'custom_low': None,
    #     'buy_trigger_time': None
    # },
    # 8
    {
        'symbol': 'TOA',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 16.20,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '16.50',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 9
    {
        'symbol': 'TISCO',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 131.50,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '133.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 10
    {
        'symbol': 'THG',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 7.65,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '7.80',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 11
    {
        'symbol': 'THAI',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 5.85,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '6.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 12
    {
        'symbol': 'TCAP',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 89.25,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '90.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 13
    {
        'symbol': 'STA',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 18.40,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '18.70',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 14
    {
        'symbol': 'SPRC',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 10.90,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '11.20',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 15
    {
        'symbol': 'SJWD',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 10.00,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '10.30',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 16
    {
        'symbol': 'SCC',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 260.00,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '263.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 17
    # {
    #     'symbol': 'RCL',
    #     'volume': None ,
    #     'amount_invested': 30000,
    #     'status': 'pending',
    #     'actual_volume_bought': 0,
    #     'await_buy_at_support': False,
    #     'trailing_stop_percentage': 0,
    #     'trailing_stop_price': 34.50,
    #     'fixed_stop_loss_percent': 0,
    #     'buy_at_price': '0',
    #     'buy_at_price1_5': '0',
    #     'buy_at_price2_5': '0',
    #     'buy2_at_price': '0',
    #     'buy_runtrend': '35.25',
    #     'sell_at_price': '0',
    #     'buy_price': 0,
    #     'custom_high': None,
    #     'custom_low': None,
    #     'buy_trigger_time': None
    # },
    # 18
    {
        'symbol': 'QH',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 1.50,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '1.53',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 19
    {
        'symbol': 'PTT',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 40.75,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '41.50',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 20
    {
        'symbol': 'PTTEP',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 150.50,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '152.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 21
    {
        'symbol': 'PR9',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 18.70,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '19.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 22
    {
        'symbol': 'MRDIYT',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 9.75,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '9.90',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 23
    {
        'symbol': 'MOSHI',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 42.00,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '42.75',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 24
    {
        'symbol': 'MASTER',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 9.05,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '9.20',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 25
    {
        'symbol': 'KTB',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 45.00,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '45.75',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 26
    {
        'symbol': 'KKP',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 112.50,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '114.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 27
    {
        'symbol': 'KBANK',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 267.00,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '270.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 28
    {
        'symbol': 'ITC',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 17.40,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '17.70',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 29
    {
        'symbol': 'IRPC',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 2.24,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '2.30',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 30
    {
        'symbol': 'HMPRO',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 6.95,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '7.10',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 31
    {
        'symbol': 'GPSC',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 51.25,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '52.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 32
    {
        'symbol': 'FORTH',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 17.00,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '17.30',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 33
    {
        'symbol': 'EASTW',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 5.55,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '5.70',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 34
    {
        'symbol': 'CHG',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 1.72,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '1.75',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 35
    {
        'symbol': 'CENTEL',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 41.75,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '42.50',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 36
    {
        'symbol': 'BTG',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 20.80,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '21.10',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 37
    {
        'symbol': 'BGRIM',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 17.60,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '17.90',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 38
    {
        'symbol': 'AURA',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 14.30,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '14.60',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 39
    {
        'symbol': 'AOT',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 66.25,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '67.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    },
    # 40
    {
        'symbol': 'ADVANC',
        'volume': None ,
        'amount_invested': 30000,
        'status': 'pending',
        'actual_volume_bought': 0,
        'await_buy_at_support': False,
        'trailing_stop_percentage': 0,
        'trailing_stop_price': 376.00,
        'fixed_stop_loss_percent': 0,
        'buy_at_price': '0',
        'buy_at_price1_5': '0',
        'buy_at_price2_5': '0',
        'buy2_at_price': '0',
        'buy_runtrend': '379.00',
        'sell_at_price': '0',
        'buy_price': 0,
        'custom_high': None,
        'custom_low': None,
        'buy_trigger_time': None
    }


]

if not stocks_to_monitor:
    print("ไม่มีหุ้นที่ถูกเพิ่มในรายการ. โปรดเพิ่มหุ้นอย่างน้อยหนึ่งตัวใน 'stocks_to_monitor' list.")
    import sys
    sys.exit()

def analyze_fibonacci_levels(data, symbol_name, custom_high=None, custom_low=None, show_output=False):
    if not data or not data.get('high') or len(data['high']) < 2 or not data.get('low') or len(data['low']) < 2:
        if show_output: print(f"ไม่พบข้อมูลหุ้น {symbol_name} เพียงพอสำหรับวิเคราะห์ Fibonacci. กรุณาลองใหม่")
        return {}

    if custom_high is not None and custom_low is not None:
        x = custom_high
        y = custom_low
        if show_output: print(f"[Fibonacci] ใช้ค่าที่กำหนดเองสำหรับ {symbol_name}: High={x:.2f}, Low={y:.2f}")
    else:
        x = data['high'][1]
        y = data['low'][1]

    if x is None or y is None or not isinstance(x, (int, float)) or not isinstance(y, (int, float)):
        if show_output: print(f"ข้อมูล High/Low ไม่ถูกต้องสำหรับหุ้น {symbol_name}. ไม่สามารถวิเคราะห์ Fibonacci ได้.")
        return {}

    diff = x - y

    z1 = x - (diff * 0.382)
    z2 = x - (diff * 0.500)
    z3 = x - (diff * 0.618)
    z4 = x - (diff * 0.786)
    z5 = x - (diff * 1.272)
    z6 = x - (diff * 1.618)
    z33 = x - (diff * 2.236)

    z44 = (z6 - z33) * 2.236
    z441 = z6 - z44
    z442 = (z6 - z33) * 1.618
    z443 = z6 - z442
    z444 = z443 - ((z443 - z441) * 2.236)

    z7 = (diff * 0.382) + y
    z8 = (diff * 0.500) + y
    z9 = (diff * 0.618) + y
    z10 = (diff * 0.786) + y
    z11 = (diff * 1.272) + y
    z12 = (diff * 1.618) + y
    z55 = (diff * 2.236) + y

    z66 = (z55 - z12) * 2.236
    z77 = (z55 - z12) * 1.618
    z88 = z12 + z77
    z99 = z12 + z66
    z100 = z88 + ((z99 - z88) * 2.236)

    if show_output:
        print(f"--- การวิเคราะห์ Fibonacci สำหรับ {symbol_name} ---")
        print(f"High: {x:.2f} | Low: {y:.2f}")
        print("[ ระดับแนวรับ (SUPPORT LEVELS) ]")
        print(f"Level 1 (0.382): {z1:.2f}")
        print(f"Level 2 (0.500): {z2:.2f}")
        print(f"Level 3 (0.618): {z3:.2f}")
        print(f"Level 4 (0.786): {z4:.2f}")
        print(f"Level 5 (1.272): {z5:.2f}")
        print(f"Level 6 (1.618): {z6:.2f}")
        print(f"  >>> III : {z33:.2f}")
        print(f"  >>> IV  : {z6:.2f}")
        print(f"  >>> V   : {z444:.2f}")

        print("[ ระดับแนวต้าน (RESISTANCE LEVELS) ]")
        print(f"Level 6 (1.618): {z12:.2f}")
        print(f"Level 5 (1.272): {z11:.2f}")
        print(f"Level 4 (0.786): {z10:.2f}")
        print(f"Level 3 (0.618): {z9:.2f}")
        print(f"Level 2 (0.500): {z8:.2f}")
        print(f"Level 1 (0.382): {z7:.2f}")
        print(f"  >>> III : {z55:.2f}")
        print(f"  >>> IV  : {z12:.2f}")
        print(f"  >>> V   : {z100:.2f}")

    return {
        'z1': z1, 'z2': z2, 'z3': z3, 'z4': z4, 'z5': z5, 'z6': z6, 'z33': z33, 'z441': z441, 'z443': z443, 'z444': z444,
        'z7': z7, 'z8': z8, 'z9': z9, 'z10': z10, 'z11': z11, 'z12': z12, 'z55': z55, 'z100': z100
    }

def is_market_open():
    thai_now = get_thai_time_object()
    if thai_now.weekday() >= 5:
        return False

    market_open_morning = thai_now.replace(hour=9, minute=55, second=0, microsecond=0)
    market_close_morning = thai_now.replace(hour=12, minute=30, second=0, microsecond=0)
    market_open_afternoon = thai_now.replace(hour=14, minute=0, second=0, microsecond=0)
    market_close_afternoon = thai_now.replace(hour=16, minute=30, second=0, microsecond=0)

    if market_open_morning <= thai_now <= market_close_morning:
        return True
    if market_open_afternoon <= thai_now <= market_close_afternoon:
        return True
    return False

def perform_trading_check(stock_info):
    global run_immediately_regardless_of_market_hours
    global R_MULTIPLE_WARNING_THRESHOLD
    global SHOW_TRADING_DECISION

    if not run_immediately_regardless_of_market_hours and not is_market_open():
        if SHOW_TRADING_DECISION: print(f"ตลาดปิดอยู่. ไม่ทำการตรวจสอบสำหรับ {stock_info['symbol']} ในขณะนี้.")
        return
    elif run_immediately_regardless_of_market_hours and not is_market_open():
        if SHOW_TRADING_DECISION: print(f"[Override] ตลาดปิดอยู่ แต่โปรแกรมถูกตั้งค่าให้ทำงานทันที. ตรวจสอบสำหรับ {stock_info['symbol']}")

    symbol_name = stock_info['symbol']
    current_status = stock_info['status']
    predefined_volume = stock_info['volume']
    amount_to_invest_baht = stock_info['amount_invested']

    custom_buy_input = stock_info.get('buy_at_price', '')
    custom_buy1_5_input = stock_info.get('buy_at_price1_5', '')
    custom_buy2_5_input = stock_info.get('buy_at_price2_5', '')
    custom_buy2_input = stock_info.get('buy2_at_price', '')
    custom_sell_input = stock_info.get('sell_at_price', '')
    buy_runtrend_input = stock_info.get('buy_runtrend', 0.0)
    custom_high_input = stock_info.get('custom_high', None)
    custom_low_input = stock_info.get('custom_low', None)

    if SHOW_TRADING_DECISION:
        print(f"\n--- กำลังทำการตรวจสอบการซื้อขายสำหรับ {symbol_name} (สถานะ: {current_status}) ณ เวลา {get_thai_time()} ---")

    try:
        res_daily = market.get_candlestick(symbol=symbol_name, interval="1d", limit=2, normalized=True)

        if (custom_high_input is None or custom_low_input is None) and \
           (not res_daily or not res_daily.get('high') or len(res_daily['high']) < 2 or res_daily['high'][1] is None or not res_daily.get('low') or len(res_daily['low']) < 2 or res_daily['low'][1] is None):
            if SHOW_TRADING_DECISION: print(f"ไม่พบข้อมูล Candlestick รายวันเพียงพอสำหรับหุ้น {symbol_name}. ไม่สามารถวิเคราะห์ได้.")
            return

        fib_levels = analyze_fibonacci_levels(res_daily, symbol_name, custom_high_input, custom_low_input, show_output=SHOW_TRADING_DECISION)
        if not fib_levels:
            return

        # Use the new helper function for getting quote
        quote = get_quote_with_retries(symbol_name)
        if not quote or 'last' not in quote or quote['last'] is None:
            if SHOW_TRADING_DECISION: print(f"ไม่พบราคาปัจจุบันสำหรับหุ้น {symbol_name}. ไม่สามารถดำเนินการได้.")
            return
        current_price_for_check = quote['last']

        res_1m = market.get_candlestick(symbol=symbol_name, interval="1m", limit=1, normalized=True)
        if not res_1m or 'close' not in res_1m or not res_1m['close'] or res_1m['close'][0] is None:
            if SHOW_TRADING_DECISION: print(f"ไม่พบข้อมูล Candlestick 1 นาทีสำหรับหุ้น {symbol_name}. ไม่สามารถดำเนินการได้.")
            return
        current_price_for_order = res_1m['close'][0]

        resolved_buy_price = fib_levels.get(custom_buy_input.lower(), float(custom_buy_input) if re.match(r'^[0-9.]+$', str(custom_buy_input)) else 0.0)
        resolved_buy1_5_price = fib_levels.get(custom_buy1_5_input.lower(), float(custom_buy1_5_input) if re.match(r'^[0-9.]+$', str(custom_buy1_5_input)) else 0.0)
        resolved_buy2_5_price = fib_levels.get(custom_buy2_5_input.lower(), float(custom_buy2_5_input) if re.match(r'^[0-9.]+$', str(custom_buy2_5_input)) else 0.0)
        resolved_buy2_price = fib_levels.get(custom_buy2_input.lower(), float(custom_buy2_input) if re.match(r'^[0-9.]+$', str(custom_buy2_input)) else 0.0)
        buy_runtrend_price = fib_levels.get(str(buy_runtrend_input).lower(), float(buy_runtrend_input) if re.match(r'^[0-9.]+$', str(buy_runtrend_input)) else 0.0)
        resolved_sell_price = fib_levels.get(custom_sell_input.lower(), float(custom_sell_input) if re.match(r'^[0-9.]+$', str(custom_sell_input)) else 0.0)

        if SHOW_TRADING_DECISION:
            print(f"\n[ การตัดสินใจซื้อขาย (TRADING DECISION) ]")
            print(f"ราคาปัจจุบัน: {current_price_for_check:.2f}")
            print(f"Buy at price: {resolved_buy_price:.2f}")
            print(f"Buy at price 1.5: {resolved_buy1_5_price:.2f}")
            print(f"Buy at price 2.5: {resolved_buy2_5_price:.2f}")
            print(f"Buy 2 at price: {resolved_buy2_price:.2f}")
            print(f"Buy runtrend: {buy_runtrend_price:.2f}")

        # --- ตรรกะการซื้อ (BUYING LOGIC) ---
        if current_status in ['pending', 'sold']:
            is_buy_triggered = False
            matched_condition = None # บันทึกว่าเข้าเงื่อนไขซื้อแบบไหน

            if stock_info['await_buy_at_support']:
                if resolved_buy_price > 0 and current_price_for_check < resolved_buy_price:
                    if SHOW_TRADING_DECISION: print(f"เข้าเงื่อนไขซื้อที่แนวรับ! ราคาปัจจุบันต่ำกว่า buy_at_price ({resolved_buy_price:.2f})")
                    is_buy_triggered = True
                    matched_condition = 'support'
                elif resolved_buy1_5_price > 0 and current_price_for_check == resolved_buy1_5_price:
                    if SHOW_TRADING_DECISION: print(f"เข้าเงื่อนไขซื้อที่แนวรับ! ราคาปัจจุบันเท่ากับ buy_at_price1_5 ({resolved_buy1_5_price:.2f})")
                    is_buy_triggered = True
                    matched_condition = 'support'
                elif resolved_buy2_5_price > 0 and current_price_for_check == resolved_buy2_5_price:
                    if SHOW_TRADING_DECISION: print(f"เข้าเงื่อนไขซื้อที่แนวรับ! ราคาปัจจุบันเท่ากับ buy_at_price2_5 ({resolved_buy2_5_price:.2f})")
                    is_buy_triggered = True
                    matched_condition = 'support'
                elif resolved_buy2_price > 0 and current_price_for_check < resolved_buy2_price:
                    if SHOW_TRADING_DECISION: print(f"เข้าเงื่อนไขซื้อที่แนวรับ! ราคาปัจจุบันต่ำกว่า buy2_at_price ({resolved_buy2_price:.2f})")
                    is_buy_triggered = True
                    matched_condition = 'support'
            else:
                if resolved_buy_price > 0 and current_price_for_check < resolved_buy_price:
                    is_buy_triggered = True
                    matched_condition = 'support'
                elif resolved_buy1_5_price > 0 and current_price_for_check == resolved_buy1_5_price:
                    is_buy_triggered = True
                    matched_condition = 'support'
                elif resolved_buy2_5_price > 0 and current_price_for_check == resolved_buy2_5_price:
                    is_buy_triggered = True
                    matched_condition = 'support'
                elif resolved_buy2_price > 0 and current_price_for_check < resolved_buy2_price:
                    is_buy_triggered = True
                    matched_condition = 'support'
                elif buy_runtrend_price > 0 and current_price_for_check > buy_runtrend_price:
                    if SHOW_TRADING_DECISION: print(f"ราคาปัจจุบันทะลุแนว Run Trend ({buy_runtrend_price:.2f}) -> เข้าเงื่อนไขซื้อตามน้ำ!")
                    is_buy_triggered = True
                    matched_condition = 'runtrend' # บันทึกว่าเข้าเงื่อนไข runtrend

            if is_buy_triggered:
                now = get_thai_time_object()
                if stock_info.get('buy_trigger_time') is None:
                    stock_info['buy_trigger_time'] = now
                    stock_info['trigger_condition'] = matched_condition # จำว่าตอนเริ่มนับถอยหลัง มาจากเงื่อนไขอะไร
                    print(f"⏱️ [Delay Logic] {symbol_name} เริ่มเข้าเงื่อนไขซื้อครั้งแรก บันทึกเวลา: {now.strftime('%H:%M:%S')} (จะเริ่มตรวจสอบและหน่วงเวลา 3 นาที)")
                    pending_buy_message = f"<b>{symbol_name}</b>: เข้าข่ายพิจารณาซื้อที่ราคา {current_price_for_check:.2f} (รอครบ 3 นาที)"
                    send_telegram_message(pending_buy_message)
                    is_buy_triggered = False
                else:
                    time_elapsed = (now - stock_info['buy_trigger_time']).total_seconds()
                    if SHOW_TRADING_DECISION: print(f"⏱️ [Delay Logic] {symbol_name} ราคายังคงอยู่ในเงื่อนไขซื้ออย่างต่อเนื่อง ผ่านไปแล้ว: {int(time_elapsed)} วินาที (เป้าหมาย 180 วินาที)")

                    if time_elapsed >= 180:
                        print(f"✅ [Delay Logic] ราคาผ่านการยืนยันเงื่อนไขครบ 3 นาทีเรียบร้อยแล้ว ยินยอมให้ยิงคำสั่งซื้อ!")
                        is_buy_triggered = True
                    else:
                        is_buy_triggered = False
            else:
                if stock_info.get('buy_trigger_time') is not None:
                    print(f"♻️ [Delay Logic] {symbol_name} ราคาดีดกลับขึ้นไปพ้นเงื่อนไขซื้อก่อนครบ 3 นาที ทำการรีเซ็ตเวลานับถอยหลังใหม่")
                    stock_info['buy_trigger_time'] = None
                    stock_info['trigger_condition'] = None # ล้างค่าที่จำไว้

            if is_buy_triggered:
                print(f"\n*** {symbol_name} เข้าเงื่อนไขซื้อและพร้อมส่งคำสั่งซื้อ! ***")

                try:
                    fresh_quote = get_quote_with_retries(symbol_name)
                    current_bid_price = fresh_quote.get('bid_price1')

                    if current_bid_price is None or current_bid_price <= 0:
                        current_bid_price = quote.get('bid_price1', current_price_for_order)
                except Exception as e:
                    logging.error(f"ไม่สามารถดึงราคา Bid สำหรับ {symbol_name} ได้: {e}")
                    current_bid_price = quote.get('bid_price1', current_price_for_order)

                # --- การกำหนดราคาตั้งซื้อ ---
                target_order_price = current_bid_price # ค่าเริ่มต้นเป็น Bid price เหมือนปกติ

                # ถ้าเงื่อนไขที่ทำให้ซื้อคือ runtrend ให้ปรับราคาตั้งซื้อไปที่ราคา runtrend แทน
                if stock_info.get('trigger_condition') == 'runtrend':
                    target_order_price = buy_runtrend_price
                    print(f"📌 [Order Info] เปลี่ยนราคาตั้งซื้อเป็นราคา Run Trend: {target_order_price:.2f}")

                quantity_to_trade = 0
                if predefined_volume is not None:
                    quantity_to_trade = predefined_volume
                elif amount_to_invest_baht is not None and amount_to_invest_baht > 0:
                    # คำนวณจำนวนหุ้นอิงจากราคาเป้าหมายที่จะตั้งซื้อจริง
                    calculated_volume = int(amount_to_invest_baht / target_order_price)
                    quantity_to_trade = (calculated_volume // 100) * 100

                if quantity_to_trade >= 100:
                    print(f"กำลังส่งคำสั่งซื้อ {symbol_name} จำนวน {quantity_to_trade} หุ้น ที่ราคา {target_order_price:.2f}...")
                    try:
                        order = equity.place_order(
                            symbol=symbol_name, side="Buy", price=target_order_price, # ใช้ตัวแปรราคาใหม่
                            volume=quantity_to_trade, pin="454545"
                        )
                        print("ส่งคำสั่งซื้อสำเร็จ:", order)
                        send_telegram_message(f"✅ <b>{symbol_name}</b>: ส่งคำสั่งซื้อสำเร็จ! จำนวน {quantity_to_trade} หุ้น ที่ราคา {target_order_price:.2f}")

                        stock_info['status'] = 'bought'
                        stock_info['actual_volume_bought'] = quantity_to_trade
                        stock_info['buy_price'] = target_order_price # อัปเดตทุนเป็นราคาที่ยิงคำสั่งไป
                        stock_info['await_buy_at_support'] = False
                        stock_info['buy_trigger_time'] = None
                        stock_info['trigger_condition'] = None # ล้างค่าหลังจากสั่งซื้อสำเร็จ

                        if stock_info['trailing_stop_percentage'] > 0:
                            stock_info['trailing_stop_price'] = target_order_price * (1 - stock_info['trailing_stop_percentage'])
                        elif stock_info.get('fixed_stop_loss_percent', 0.0) > 0:
                            stock_info['trailing_stop_price'] = target_order_price * (1 - stock_info['fixed_stop_loss_percent'])
                    except Exception as e:
                        print(f"ซื้อไม่สำเร็จสำหรับ {symbol_name}: {e}")
                        send_telegram_message(f"❌ <b>{symbol_name}</b>: ซื้อไม่สำเร็จ! ({e})")
                else:
                    print(f"ปริมาณหุ้นไม่เพียงพอที่จะส่งคำสั่งซื้อขั้นต่ำ 100 หุ้น")

        # --- ตรรกะการขาย (SELLING LOGIC) ---
        elif current_status == 'bought':
            if stock_info['buy_price'] > 0 and stock_info['actual_volume_bought'] > 0:
                profit_loss_per_share = current_price_for_check - stock_info['buy_price']
                total_profit_loss = profit_loss_per_share * stock_info['actual_volume_bought']

                initial_risk_1R_total = 0
                if stock_info['trailing_stop_percentage'] > 0:
                    initial_risk_1R_total = stock_info['buy_price'] * stock_info['actual_volume_bought'] * stock_info['trailing_stop_percentage']
                elif stock_info.get('fixed_stop_loss_percent', 0.0) > 0:
                    initial_risk_1R_total = stock_info['buy_price'] * stock_info['actual_volume_bought'] * stock_info['fixed_stop_loss_percent']

                if SHOW_TRADING_DECISION:
                    print(f"  [P/L ปัจจุบัน] ทุน: {stock_info['buy_price']:.2f}, ปัจจุบัน: {current_price_for_check:.2f}, หุ้นในพอร์ต: {stock_info['actual_volume_bought']}")
                    print(f"  กำไร/ขาดทุนรวม: {total_profit_loss:.2f} บาท")

                if initial_risk_1R_total > 0 and total_profit_loss <= -initial_risk_1R_total * R_MULTIPLE_WARNING_THRESHOLD:
                    print(f"  !!! คำเตือน: {symbol_name} ขาดทุนใกล้ขีดจำกัด -1R (ปัจจุบัน {-total_profit_loss:.2f} บาท จากค่าวิกฤต {-initial_risk_1R_total:.2f} บาท) !!!")

            if stock_info['trailing_stop_percentage'] > 0:
                calculated_trailing = current_price_for_check * (1 - stock_info['trailing_stop_percentage'])
                if calculated_trailing > stock_info['trailing_stop_price']:
                    stock_info['trailing_stop_price'] = calculated_trailing
                    print(f"  [Trailing Stop ขยับขึ้น] จุด Stop Loss ใหม่ของ {symbol_name} คือ {stock_info['trailing_stop_price']:.2f}")

            execute_sell = False
            sell_reason = ""

            if resolved_sell_price > 0 and current_price_for_check >= resolved_sell_price:
                execute_sell = True
                sell_reason = f"ราคาปัจจุบัน ({current_price_for_check:.2f}) ถึงเป้าหมายขายทำกำไรที่กำหนด ({resolved_sell_price:.2f})"
            elif stock_info['trailing_stop_price'] > 0 and current_price_for_check <= stock_info['trailing_stop_price']:
                execute_sell = True
                sell_reason = f"ราคาปัจจุบัน ({current_price_for_check:.2f}) ต่ำกว่าหรือชนจุด Stop Loss ระบบล็อกความเสี่ยงทันที ({stock_info['trailing_stop_price']:.2f})"

            if execute_sell:
                quantity_to_sell = stock_info.get('actual_volume_bought', 0)
                if quantity_to_sell >= 100:
                    print(f"🚀 [TRIGGER SELL] {symbol_name} {sell_reason}")
                    print(f"กำลังส่งคำสั่งขาย {symbol_name} จำนวน {quantity_to_sell} หุ้น ที่ราคา {current_price_for_order:.2f}...")
                    try:
                        order = equity.place_order(
                            symbol=symbol_name, side="Sell", price=current_price_for_order,
                            volume=quantity_to_sell, pin="454545"
                        )
                        print("ส่งคำสั่งขายสำเร็จ! ผลลัพธ์ API:", order)
                        send_telegram_message(f"🚀 <b>{symbol_name}</b>: ขายแล้ว! {sell_reason}")

                        stock_info['status'] = 'pending'
                        stock_info['actual_volume_bought'] = 0
                        stock_info['await_buy_at_support'] = True
                        stock_info['trailing_stop_price'] = 0.0
                        stock_info['buy_price'] = 0.0
                    except Exception as e:
                        print(f"❌ เกิดข้อผิดพลาดจากทางตลาด ไม่สามารถส่งคำสั่งขาย {symbol_name} ได้: {e}")
                        send_telegram_message(f"❌ <b>{symbol_name}</b>: ส่งคำสั่งขายไม่สำเร็จ! ({e})")
                else:
                    print(f"ไม่สามารถส่งขายได้เนื่องจากจำนวนหุ้นในตัวแปรบอตไม่ถูกต้อง (จำนวน: {quantity_to_sell})")
            else:
                if SHOW_TRADING_DECISION:
                    print(f"  สถานะ: ถือหุ้นรอเงื่อนไขถัดไป (Stop Loss ปัจจุบัน: {stock_info['trailing_stop_price']:.2f} | เป้าหมายทำกำไร: {resolved_sell_price if resolved_sell_price > 0 else 'ไม่ได้ตั้งไว้'})")

    except Exception as e:
        print(f"เกิดข้อผิดพลาดในระหว่างการตรวจสอบ {symbol_name}: {e}")

# ==============================================================================
# --- ส่วนควบคุมการเริ่มต้นและรันระบบทำงาน (Parallel Processing) ---
# ==============================================================================
print(f"กำลังรอเวลาเปิดตลาด หรือหลังเปิดตลาด {startup_delay_seconds_after_market_open} วินาที...")

market_open_time = get_thai_time_object().replace(hour=9, minute=55, second=0, microsecond=0)
market_start_time_with_delay = market_open_time + datetime.timedelta(seconds=startup_delay_seconds_after_market_open)
current_thai_time_obj = get_thai_time_object()

if not run_immediately_regardless_of_market_hours and current_thai_time_obj < market_start_time_with_delay:
    time_to_wait = (market_start_time_with_delay - current_thai_time_obj).total_seconds()
    if time_to_wait > 0:
        print(f"เหลือเวลาอีก {int(time_to_wait)} วินาที... เพื่อเริ่มการตรวจสอบ...")
        time.sleep(time_to_wait)
else:
    print("เริ่มระบบทำงานทันทีตามค่า Override ด้านบน")

print(f"เริ่มระบบเฝ้าระวังและแสกนราคาตลาดแบบ Parallel Real-time ทุก 1 วินาที... (กด Ctrl+C เพื่อหยุด)")

def scan_all_stocks_parallel():
    clear_output(wait=True)
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(stocks_to_monitor)) as executor:
        executor.map(perform_trading_check, stocks_to_monitor)
    # Clear output every 3 minutes
    if int(time.time()) % 180 == 0:
        clear_output(wait=True)

schedule.clear()

schedule.every(5).seconds.do(scan_all_stocks_parallel)
# Schedule the download function to run daily at 16:35
schedule.every().day.at("16:35").do(download_resources_file)
logging.info("ตั้งค่าการดาวน์โหลดไฟล์ทรัพยากรทุกวันเวลา 16:35 น.")

# Schedule the bot to stop daily at 12:31
schedule.every().day.at("12:32").do(stop_bot_gracefully)
logging.info("ตั้งค่าการหยุดการทำงานของบอตทุกวันเวลา 12:32 น.")

# Schedule the bot to stop daily at 16:32
schedule.every().day.at("16:32").do(stop_bot_gracefully)
logging.info("ตั้งค่าการหยุดการทำงานของบอตทุกวันเวลา 16:32 น.")

while not stop_bot_flag:
    try:
        schedule.run_pending()
        time.sleep(1)
    except KeyboardInterrupt:
        print("\nหยุดการทำงานของบอตเรียบร้อยแล้วโดยผู้ใช้งาน.")
        break
print("บอตหยุดการทำงานแล้ว.")